# Two-phase asset identification: the final notebook

The working version of the two-phase prompt, `two_phase_p1o_first`, on the 41 NEORV32 modules with gpt-5.6-luna at
reasoning effort high. Phase 1 names the conceptual assets with the four CSA questions; Phase 2 walks the block diagram
and names the structural assets of each concept by points of influence and observation (PIO). Two calls per module.

What the cells do:

1. Setup and checks (cells 1 to 3): self-tests, the prompt check, the cost estimate. Nothing is sent.
2. A pilot on two modules (cell 4), the full run (cell 7), repeat runs (cell 8) and the three runs in one worker pool
   (cell 9). Every run flag is `False`; set one to `True` to send. A stored reply is never asked again, so a stopped run
   resumes where it was and nothing is paid twice.
3. Scoring (cell 10): the exact scorer over the stored replies, tuning 15, held-out 26 and all 41, next to LAsset's
   published initial lists. No call is made.

Files: the kit is `experiments/03_two_phase_csa_pio/lab/` (code; built inputs `inputs/v1/`; prompts `prompts/two_phase_<version>/`, one README each).
Replies go to `experiments/03_two_phase_csa_pio/lab/runs/<version>__v1/<model>__<effort>/r<k>/` (`phase1/`, `phase2/`, `assets/`). The three runs of the working version and of `two_phase_v7` are
published; the replies of the other versions are local. The API key is read from `API.env` in the repository folder.


In [ ]:
# 1. Setup (no call is made)
import json, os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "experiments" / "03_two_phase_csa_pio" / "lab" / "chain.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)                         # the scorer reads data/ relative to the working folder
LAB = ROOT / "experiments" / "03_two_phase_csa_pio" / "lab"
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))

# Built inputs: "v1" (all runs so far) or "v1k" (v1 with an exact diagram key line; for the next prompt version).
# Set before the imports; runs go to runs/<prompt version>__<input version>/, so the two never mix.
INPUT_VERSION = "v1"
os.environ["PIO_INPUT_VERSION"] = INPUT_VERSION

import importlib
import labkit as K, check_lab_prompt as CK, chain as CH, cost_estimate as CE
for _m in (K, CK, CH, CE):
    importlib.reload(_m)

VERSION = "two_phase_p1o_first"            # the working version (user decision, 2026-10-19)
MODEL, EFFORT, REP = "gpt-5.6-luna", "high", 0
PILOT = ["neorv32_wdt", "neorv32_bus"]
ALL = K.gt_modules()["tuning"] + K.gt_modules()["heldout"]   # all 41 modules (15 tuning + 26 held-out)
OUT = CH.run_dir(VERSION, MODEL, REP, EFFORT)
print("repository:", ROOT)
print("inputs:", K.INPUTS)
print("run folder:", OUT)
print("API.env present:", (ROOT / "API.env").exists() or (ROOT.parent / "API.env").exists())

In [ ]:
# 2. Checks: runner self-test, extra tests (fake OpenAI client), prompt check. Nothing is sent.
import subprocess
assert CH.selftest(), "chain self-test failed"
r = subprocess.run([sys.executable, str(LAB / "chain_tests.py")], capture_output=True, text=True, encoding="utf-8")
print(r.stdout[-2500:] or r.stderr[-2500:])
assert r.returncode == 0, "chain_tests failed"
assert CK.check(VERSION), "prompt check failed"

In [ ]:
# 3. Cost estimate before sending anything (the output size is borrowed from an earlier prompt)
print("== pilot")
CE.estimate(VERSION, PILOT)
print("\n== full run, 41 modules")
CE.estimate(VERSION, ALL)

In [ ]:
# 4. The pilot: 2 modules x 2 phases = 4 calls. Set RUN_PILOT = True to send.
RUN_PILOT = False

if RUN_PILOT:
    OUT.mkdir(parents=True, exist_ok=True)
    call = CH.openai(MODEL, EFFORT, out=OUT)
    rows = CH.run(VERSION, PILOT, call, MODEL, EFFORT, REP, executor="openai", workers=2)
else:
    print("RUN_PILOT is False: nothing sent")

In [ ]:
# 5. What the pilot cost, and the full run re-estimated from the pilot's measured output
res = CE.reestimate(VERSION, MODEL, REP, EFFORT, pilot_modules=PILOT)

In [ ]:
# 6. Pilot readings on the two modules (a pilot, not a result): checks, concepts, points, primaries, P and R
print(json.dumps(CH.readings(VERSION, MODEL, REP, PILOT, EFFORT), indent=1, default=str))
for m in PILOT:
    for ph in (1, 2):
        f = OUT / f"phase{ph}" / f"{m}.json"
        if f.exists():
            rec = json.loads(f.read_text(encoding="utf-8"))
            print(f"{m} phase {ph}: {len(rec.get('problems', []))} problems", rec.get("problems", [])[:5])

In [ ]:
# 7. Optional: the full run on all 41 modules (the pilot's stored replies are reused). Set RUN_FULL = True to send.
RUN_FULL = False

if RUN_FULL:
    OUT.mkdir(parents=True, exist_ok=True)
    call = CH.openai(MODEL, EFFORT, out=OUT)
    rows = CH.run(VERSION, ALL, call, MODEL, EFFORT, REP, executor="openai", workers=13)
    print(json.dumps(CH.readings(VERSION, MODEL, REP, ALL, EFFORT), indent=1, default=str))
else:
    print("RUN_FULL is False: nothing sent")

In [ ]:
# 8. Repeat runs r1 and r2 of the same model (r0 is done), with the same settings as cell 7.
#    A reply that fails the format check is moved to _schema_failed/ and asked once more, as was done for r0.
import shutil
RUN_REPEATS = False
REPS = [0, 1, 2]

if RUN_REPEATS:
    for rep in REPS:
        out = CH.run_dir(VERSION, MODEL, rep, EFFORT)
        out.mkdir(parents=True, exist_ok=True)
        call = CH.openai(MODEL, EFFORT, out=out)
        rows = CH.run(VERSION, ALL, call, MODEL, EFFORT, rep, executor="openai", workers=15)
        retry = []
        for r in rows:
            if "schema failure" not in str(r.get("status")):
                continue
            m, ph = r["module"], (1 if "phase 1" in r["status"] else 2)
            dest = out / "_schema_failed" / f"phase{ph}"
            if (dest / f"{m}.json").exists():          # it already had its one retry
                print(f"r{rep} {m} phase {ph}: failed again after its retry; left as is")
                continue
            dest.mkdir(parents=True, exist_ok=True)
            shutil.move(str(out / f"phase{ph}" / f"{m}.json"), str(dest / f"{m}.json"))
            with open(out / "_schema_failed" / "README.txt", "a", encoding="utf-8") as fh:
                fh.write(f"{m} phase {ph}: the first reply failed the format check; moved here and asked once more.\n")
            retry.append(m)
        if retry:
            CH.run(VERSION, retry, call, MODEL, EFFORT, rep, executor="openai", workers=15)
        print(f"--- r{rep}:", json.dumps(CH.readings(VERSION, MODEL, rep, ALL, EFFORT), indent=1, default=str))
else:
    print("RUN_REPEATS is False: nothing sent")

In [ ]:
# 9. The three runs in ONE worker pool (chain.run_many). Every (version, rep, module) pair starts at once, largest
#    modules first, so three runs take about the time of one. A 429 rate-limit refusal is sent again after a wait; a
#    reply that fails the format check is moved to _schema_failed/ and asked once more inside its worker. Stored
#    replies are never asked again, so re-running this cell only collects what is missing. The preview shows what
#    would be sent; nothing is sent while RUN_MANY is False. To stop a run: press stop ONCE and wait until the cell
#    ends; then run the cell again and the requests already sent are collected, none paid twice.
#    Afterwards: python experiments/03_two_phase_csa_pio/lab/analysis/score_versions.py
importlib.reload(K); importlib.reload(CK); importlib.reload(CH)
RUN_MANY = False                                # preview only; set True to send
JOBS = [(VERSION, r) for r in (0, 1, 2)]
ALL = K.gt_modules()["tuning"] + K.gt_modules()["heldout"]   # all 41 modules, as in cell 1
WORKERS = None                                   # every (version, rep, module) pair at once (here 3 x 41 = 123)
PHASE1_REUSED = set()                            # versions that must find all their Phase 1 replies stored


def preview(jobs):
    """From the stored files only (nothing is sent): per run, the replies already stored for the same messages and
    the calls still to send. A stored Phase 1 reply asked with another message is reported, as it would stop the run.
    A reply that fails the format check is asked once more (one retry per module), so the ceiling is one more call per
    module with anything to send."""
    total = pairs = 0
    for v, rep in jobs:
        out = CH.run_dir(v, MODEL, rep, EFFORT)
        n = {"phase 1 stored": 0, "phase 1 to send": 0, "phase 2 stored": 0, "phase 2 to send (at most)": 0}
        for m in ALL:
            f1, f2 = out / "phase1" / f"{m}.json", out / "phase2" / f"{m}.json"
            if f1.exists():
                r1 = json.loads(f1.read_text(encoding="utf-8"))
                want = (CH.sha12(CK.system(v, 1)), CH.sha12(CH.user_message(v, 1, m)))
                if (r1.get("system sha"), r1.get("user sha")) != want:
                    raise RuntimeError(f"{f1} was asked with another message; the run would stop this module")
                n["phase 1 stored"] += 1
            else:
                n["phase 1 to send"] += 1
            n["phase 2 stored" if f2.exists() else "phase 2 to send (at most)"] += 1
            pairs += not (f1.exists() and f2.exists())
        if v in PHASE1_REUSED and n["phase 1 to send"]:
            raise RuntimeError(f"{v} r{rep} lacks {n['phase 1 to send']} of its copied Phase 1 replies: run "
                               f"analysis/prompt_builds/prepare_phase1_copy.py {v} [--from=two_phase_v6abc for v6d] --apply")
        total += n["phase 1 to send"] + n["phase 2 to send (at most)"]
        print(f"  {v} r{rep}: {n}")
    print(f"  calls to send: {total} if every reply passes the format check; at most {total + pairs} with the one retry "
          "per module")


print("preview (nothing is sent):")
preview(JOBS)
if RUN_MANY:
    for v in sorted({v for v, _ in JOBS}):
        assert CK.check(v), f"prompt check failed: {v}"
    res = CH.run_many(JOBS, ALL, lambda out: CH.openai(MODEL, EFFORT, out=out, poll=5), MODEL, EFFORT,
                      executor="openai", workers=WORKERS)
    for v, rep in JOBS:
        print(f"--- {v} r{rep}:", json.dumps(CH.readings(v, MODEL, rep, ALL, EFFORT), indent=1, default=str))
        if v in PHASE1_REUSED:
            print("  note: 'usage of stored replies' includes the copied Phase 1 replies, paid in an earlier run; this "
                  "version's own cost is compare_condensed.py's 'cost USD (Phase 2 only)'")
else:
    print("RUN_MANY is False: nothing sent")

In [ ]:
# 10. Scoring from the stored replies (no call): the exact scorer, every version, then LAsset's published lists.
#     Precision = correct / listed; recall = correct / findable (109 tuning, 187 held-out, 296 all 41 per run); pooled over runs.
import subprocess
for script in ("analysis/score_versions.py", "analysis/score_lasset_exact.py"):
    r = subprocess.run([sys.executable, str(LAB / script)], capture_output=True, text=True, encoding="utf-8", cwd=ROOT)
    assert r.returncode == 0, r.stderr[-2000:]
    lines = r.stdout.splitlines()
    keep = [l for l in lines if l.startswith("| p1o_first") or l.startswith("| v9j") or l.startswith("| v1 ")
            or l.startswith("LAsset") or "self-test" in l or l.startswith("| version")]
    print("\n".join(keep))
# the working version's per-run counts, as the README's tables read them
rec = json.loads((LAB / "analysis" / "score_versions.json").read_text(encoding="utf-8"))["two_phase_p1o_first"]
for split, d in rec.items():
    print(f"{split:8s} P {d['P']:.3f}  R {d['R']:.3f}  per run (correct/listed) {d['per_run']}  findable {d['findable']}")
